# WORKSHOP: Weather Measurements Pipeline - Initial Configuration

Este script configura la infraestructura necesaria para el proyecto de medición del clima usando Unity Catalog en Databricks. Cada sección debe ejecutarse en una celda separada para facilitar el seguimiento durante el workshop.

El script es idempotente: puede ejecutarse varias veces sin causar errores.

## CREAR EL CATÁLOGO

Crea el catálogo principal para los datos meteorológicos.

Verifica que el catálogo se haya creado correctamente.

In [0]:
%sql CREATE CATALOG IF NOT EXISTS weather
COMMENT 'Catalog for weather data - Contains current, historical, and forecast measurements';

SHOW CATALOGS LIKE 'weather';

Los catálogos en Databricks Unity Catalog son contenedores lógicos para organizar esquemas y tablas,
permitiendo una gestión centralizada de datos, permisos y gobernanza en toda la plataforma.

Beneficios de Unity Catalog:
1. Seguridad centralizada: Control de acceso granular a nivel de catálogo, esquema y tabla.
2. Gobernanza de datos: Trazabilidad, auditoría y cumplimiento normativo.
3. Descubrimiento de datos: Facilita la búsqueda y organización de activos de datos.
4. Multicloud y multiworkspace: Permite compartir datos entre diferentes workspaces y nubes.
5. Integración con herramientas externas: Compatible con BI, ML y otros servicios.


## USAR EL CATÁLOGO

Establece el contexto en el nuevo catálogo.

Verifica el catálogo actual.

In [0]:
%sql USE CATALOG weather;

SELECT current_catalog() as current_catalog;


## CREAR SCHEMA

Esquema 1: Medidas en tiempo real (actuales).

In [0]:
%sql CREATE SCHEMA IF NOT EXISTS forecast_and_current_weather
COMMENT 'Schema for current weather measurements in real-time - Contains Bronze and Silver pipeline tables';

## CREAR TABLAS DE SOPORTE

Tabla: Puntos de medición de OpenMeteo.

Verifica que la tabla se haya creado correctamente.

La primera tabla de soporte, `openmeteo_measurement_points`, se crea vacía, es decir, no contiene datos inicialmente. Esta tabla define la estructura para almacenar los puntos de medición de OpenMeteo, pero aún no se han insertado registros.

In [0]:
%sql CREATE TABLE IF NOT EXISTS weather.forecast_and_current_weather.openmeteo_measurement_points (
    latitude DOUBLE,
    longitude DOUBLE,
    display_name STRING COLLATE UTF8_BINARY,
    place_name STRING COLLATE UTF8_BINARY,
    city STRING COLLATE UTF8_BINARY,
    province STRING COLLATE UTF8_BINARY,
    country STRING COLLATE UTF8_BINARY
  );

SHOW TABLES IN weather.forecast_and_current_weather LIKE 'openmeteo_measurement_points';

## CARGAR DATOS DE UBICACIONES

La siguiente celda carga los datos de ubicaciones desde un archivo CSV y los almacena en una tabla de Unity Catalog.

**Proceso:**

1. **Obtener el usuario actual:** Se consulta `current_user()` para construir la ruta dinámica al archivo CSV
2. **Leer el archivo CSV:** Usa `spark.read.csv()` con:
   - `header=True`: La primera fila contiene los nombres de las columnas
   - `inferSchema=False`: No infiere tipos automáticamente, todas las columnas son STRING
3. **Guardar como tabla:** Usa `.write.mode("overwrite")` para:
   - Sobrescribir la tabla si ya existe
   - Crear la tabla `weather.forecast_and_current_weather.locations` en Unity Catalog

**Tabla resultante:** `weather.forecast_and_current_weather.locations`

In [0]:
username = spark.sql("SELECT current_user()").collect()[0][0]
spark.read.csv(
    f"/Workspace/Users/{username}/openmeteo-databricks-pipeline/data/locations.csv",
    header=True,
    inferSchema=False,
).write.mode("overwrite").saveAsTable(
    "weather.forecast_and_current_weather.locations"
)


## CREAR TABLA DE CÓDIGOS METEOROLÓGICOS

La siguiente celda crea la tabla `weather_codes` que mapea los códigos numéricos de clima (que devuelve la API de OpenMeteo) a descripciones legibles.

**Proceso:**

1. **Importar Row:** `from pyspark.sql import Row` - clase para crear filas de datos estructurados
2. **Definir lista de códigos:** Se crea una lista de objetos `Row` con:
   - `weather_code`: Código numérico (0-99) según la especificación WMO
   - `weather_category`: Categoría general (ej: "Rain", "Snow", "Fog")
   - `weather_intensity`: Nivel de intensidad (ej: "Light", "Moderate", "Heavy")
3. **Crear DataFrame:** `spark.createDataFrame(weather_codes)` convierte la lista en un DataFrame de Spark
4. **Guardar como tabla:** `.write.mode("ignore")` crea la tabla en Unity Catalog

### Modo de Escritura: `mode("ignore")`

La celda usa `.write.mode("ignore")` para guardar la tabla. Este modo tiene un comportamiento especial:

* **Si la tabla NO existe:** La crea con los datos del DataFrame
* **Si la tabla YA existe:** NO hace nada - no sobrescribe, no agrega datos, no lanza error

**Comparación con otros modos:**

| Modo | Comportamiento |
| --- | --- |
| `ignore` | No hace nada si la tabla existe (usado aquí) |
| `overwrite` | Borra y recrea la tabla completamente |
| `append` | Agrega nuevos registros a la tabla existente |
| `error` o `errorifexists` | Lanza error si la tabla ya existe (modo por defecto) |

**¿Por qué usar `ignore` aquí?**

Porque los códigos meteorológicos son estándares WMO que no cambian. Una vez creada la tabla en la primera ejecución, las ejecuciones posteriores del script no necesitan recrearla ni modificarla. Esto hace el script **idempotente** - puede ejecutarse múltiples veces sin causar errores ni duplicar datos.

**Uso de la tabla:**

Esta tabla se usará como tabla de dimensión para hacer `JOIN` con las tablas de mediciones meteorológicas, permitiendo convertir códigos como `61` (que viene en la API) a descripciones legibles como "Rain - Slight intensity".

**Ejemplo de uso:**
```sql
SELECT 
  measurement.*, 
  wc.weather_category, 
  wc.weather_intensity
FROM silver_current_weather measurement
LEFT JOIN weather_codes wc ON measurement.weather_code = wc.weather_code
```

**Tabla resultante:** `weather.forecast_and_current_weather.weather_codes`

---



In [0]:
from pyspark.sql import Row

weather_codes = [
    Row(weather_code=0, weather_category="Clear sky", weather_intensity=None),
    Row(weather_code=1, weather_category="Cloud cover", weather_intensity="Mainly clear"),
    Row(weather_code=2, weather_category="Cloud cover", weather_intensity="Partly cloudy"),
    Row(weather_code=3, weather_category="Cloud cover", weather_intensity="Overcast"),
    Row(weather_code=45, weather_category="Fog", weather_intensity="Fog"),
    Row(weather_code=48, weather_category="Fog", weather_intensity="Depositing rime fog"),
    Row(weather_code=51, weather_category="Drizzle", weather_intensity="Light intensity"),
    Row(weather_code=53, weather_category="Drizzle", weather_intensity="Moderate intensity"),
    Row(weather_code=55, weather_category="Drizzle", weather_intensity="Dense intensity"),
    Row(weather_code=56, weather_category="Freezing Drizzle", weather_intensity="Light intensity"),
    Row(weather_code=57, weather_category="Freezing Drizzle", weather_intensity="Dense intensity"),
    Row(weather_code=61, weather_category="Rain", weather_intensity="Slight intensity"),
    Row(weather_code=63, weather_category="Rain", weather_intensity="Moderate intensity"),
    Row(weather_code=65, weather_category="Rain", weather_intensity="Heavy intensity"),
    Row(weather_code=66, weather_category="Freezing Rain", weather_intensity="Light intensity"),
    Row(weather_code=67, weather_category="Freezing Rain", weather_intensity="Heavy intensity"),
    Row(weather_code=71, weather_category="Snow fall", weather_intensity="Slight intensity"),
    Row(weather_code=73, weather_category="Snow fall", weather_intensity="Moderate intensity"),
    Row(weather_code=75, weather_category="Snow fall", weather_intensity="Heavy intensity"),
    Row(weather_code=77, weather_category="Snow grains", weather_intensity=None),
    Row(weather_code=80, weather_category="Rain showers", weather_intensity="Slight intensity"),
    Row(weather_code=81, weather_category="Rain showers", weather_intensity="Moderate intensity"),
    Row(weather_code=82, weather_category="Rain showers", weather_intensity="Violent intensity"),
    Row(weather_code=85, weather_category="Snow showers", weather_intensity="Slight intensity"),
    Row(weather_code=86, weather_category="Snow showers", weather_intensity="Heavy intensity"),
    Row(weather_code=95, weather_category="Thunderstorm", weather_intensity="Slight or moderate"),
    Row(weather_code=96, weather_category="Thunderstorm with hail", weather_intensity="Slight hail"),
    Row(weather_code=99, weather_category="Thunderstorm with hail", weather_intensity="Heavy hail"),
]

df_weather_codes = spark.createDataFrame(weather_codes)
df_weather_codes.write.mode("ignore").saveAsTable("weather.forecast_and_current_weather.weather_codes")

## CREAR FUNCIONES SQL

**Función:** get_wind_direction

**Propósito:** Convierte la dirección del viento en grados (0-360) a dirección cardinal.

**Uso:** `SELECT get_wind_direction(wind_direction_10m) FROM silver_current_weather`

In [0]:
%sql CREATE FUNCTION IF NOT EXISTS forecast_and_current_weather.get_wind_direction(degrees DOUBLE)
  RETURNS STRING
  LANGUAGE SQL
  COMMENT 'Convierte grados (0-360) a dirección cardinal: Norte, Sur, Este, Oeste'
  RETURN CASE
    WHEN degrees IS NULL THEN NULL
    WHEN (degrees >= 315 OR degrees < 45) THEN 'N'
    WHEN (degrees >= 45 AND degrees < 135) THEN 'E'
    WHEN (degrees >= 135 AND degrees < 225) THEN 'S'
    WHEN (degrees >= 225 AND degrees < 315) THEN 'O'
    ELSE NULL
  END;

## FORMAS DE CREAR FUNCIONES EN SPARK Y UNITY CATALOG

### 1. Funciones SQL en Unity Catalog (Como `get_wind_direction`)

Son funciones **persistentes** almacenadas en Unity Catalog que pueden ser reutilizadas en cualquier notebook, query o dashboard.

```sql
CREATE FUNCTION schema.function_name(param1 TYPE, param2 TYPE)
  RETURNS RETURN_TYPE
  LANGUAGE SQL
  COMMENT 'Descripción de la función'
  RETURN <expresión SQL>;
```

**Características:**
* ✅ **Persistentes**: Se guardan en Unity Catalog
* ✅ **Compartibles**: Disponibles en todo el workspace
* ✅ **Gobernanza**: Control de acceso con permisos UC
* ✅ **Lenguaje**: SQL puro
* ✅ **Rendimiento**: Optimizadas por Catalyst
* ❌ **Limitación**: Solo lógica SQL, no código Python/Scala

**Ejemplo:**
```sql
CREATE FUNCTION weather.utils.celsius_to_fahrenheit(celsius DOUBLE)
  RETURNS DOUBLE
  RETURN (celsius * 9/5) + 32;
```

---

### 2. UDFs Python (User Defined Functions)

Funciones Python que se registran **temporalmente** en la sesión de Spark.

```python
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

@udf(returnType=StringType())
def classify_temperature(temp):
    if temp < 0:
        return "Freezing"
    elif temp < 15:
        return "Cold"
    elif temp < 25:
        return "Mild"
    else:
        return "Hot"

# Usar en DataFrame
df.withColumn("temp_category", classify_temperature(df.temperature))
```

**Características:**
* ✅ **Flexibilidad**: Lógica compleja en Python
* ✅ **Librerías**: Acceso a bibliotecas Python (numpy, pandas, etc.)
* ❌ **Temporales**: Solo existen en la sesión actual
* ❌ **Rendimiento**: Más lentas (serialización Python-JVM)
* ❌ **No persistentes**: No se guardan en Unity Catalog

---

### 3. UDFs Python Persistentes en Unity Catalog

Funciones Python que **SÍ se guardan** en Unity Catalog (Databricks Runtime 13.3+).

```python
from pyspark.sql.types import StringType

spark.udf.register(
    "schema.classify_temp_python",
    lambda temp: "Hot" if temp > 25 else "Cold",
    StringType()
)

# Crear como función UC persistente
spark.sql("""
  CREATE FUNCTION IF NOT EXISTS schema.classify_temp_python(temp DOUBLE)
  RETURNS STRING
  LANGUAGE PYTHON
  AS $$
    if temp < 0:
        return "Freezing"
    elif temp < 15:
        return "Cold" 
    elif temp < 25:
        return "Mild"
    else:
        return "Hot"
  $$
""")
```

**Características:**
* ✅ **Persistentes**: Se guardan en Unity Catalog
* ✅ **Compartibles**: Disponibles en todo el workspace
* ✅ **Gobernanza**: Control de acceso UC
* ✅ **Flexibilidad**: Lógica Python compleja
* ❌ **Rendimiento**: Más lentas que SQL nativas

---

### 4. Pandas UDFs (Vectorized UDFs)

Funciones Python optimizadas que procesan datos en lotes usando pandas.

```python
from pyspark.sql.functions import pandas_udf
import pandas as pd

@pandas_udf("double")
def vectorized_multiply(temps: pd.Series) -> pd.Series:
    return temps * 1.8 + 32

df.withColumn("temp_fahrenheit", vectorized_multiply(df.temperature))
```

**Características:**
* ✅ **Rendimiento**: Mucho más rápidas que UDFs normales
* ✅ **Procesamiento vectorizado**: Opera sobre columnas completas
* ✅ **Pandas**: Usa operaciones vectorizadas de pandas
* ❌ **Temporales**: No persisten en Unity Catalog

---

### 5. Funciones Scala (UDFs)

Funciones escritas en Scala, compiladas y ejecutadas en la JVM.

```scala
import org.apache.spark.sql.functions.udf

val classifyTemp = udf((temp: Double) => {
  if (temp < 0) "Freezing"
  else if (temp < 15) "Cold"
  else if (temp < 25) "Mild" 
  else "Hot"
})

df.withColumn("temp_category", classifyTemp($"temperature"))
```

**Características:**
* ✅ **Rendimiento**: Muy rápidas (nativas JVM)
* ✅ **Type-safe**: Verificación de tipos en compilación
* ❌ **Temporales**: No persisten por defecto
* ❌ **Complejidad**: Requiere conocimiento de Scala

---

### COMPARATIVA GENERAL

| Tipo de función | Gestionada por Unity Catalog | Lenguaje | Uso principal |
| --- | --- | --- | --- |
| **SQL UC Function** | ✅ Sí | SQL | Transformaciones SQL simples y reutilizables |
| **Python UC Function** | ✅ Sí | Python | Lógica compleja persistente |
| **UDF Python** | ❌ No (temporal) | Python | Transformaciones ad-hoc en notebooks |
| **Pandas UDF** | ❌ No (temporal) | Python + Pandas | Procesamiento vectorizado de alto rendimiento |
| **UDF Scala** | ❌ No (temporal) | Scala | Máximo rendimiento en JVM |

---

### RECOMENDACIONES

1. **Usa SQL UC Functions** cuando:
   - La lógica es expresable en SQL
   - Necesitas compartir la función entre equipos
   - Quieres gobernanza y control de acceso
   - Ejemplo: Conversiones, clasificaciones simples, cálculos matemáticos

2. **Usa Python UC Functions** cuando:
   - Necesitas lógica compleja no expresable en SQL
   - Quieres persistencia y compartir código
   - La función usa librerías Python
   - Ejemplo: Machine learning scoring, procesamiento de texto complejo

3. **Usa Pandas UDFs** cuando:
   - Necesitas alto rendimiento en Python
   - Procesas grandes volúmenes de datos
   - La función es temporal (solo para análisis actual)
   - Ejemplo: Transformaciones numéricas masivas

4. **Usa UDFs Python simples** cuando:
   - Prototipas en un notebook
   - La función es de un solo uso
   - No necesitas persistencia
   - Ejemplo: Exploración de datos rápida

---

### EN ESTE PROYECTO

Usamos **SQL UC Functions** porque:
* ✅ Las transformaciones son simples (conversión de grados a cardinal)
* ✅ Queremos que estén disponibles en dashboards y queries
* ✅ Necesitamos gobernanza y control de acceso
* ✅ Optimización automática de Spark
* ✅ No requieren dependencias externas
## Formas de crear funciones en Spark y Unity Catalog

En Databricks y Spark, existen dos formas principales de crear funciones personalizadas:

### 1. Funciones SQL (SQL User-Defined Functions, UDFs)

Estas funciones se crean directamente usando SQL y pueden ser gestionadas por Unity Catalog. Ejemplo:

sql
CREATE FUNCTION IF NOT EXISTS catalog.schema.function_name(param TYPE)
RETURNS TYPE
LANGUAGE SQL
RETURN <expresión_sql>;


- **Ventajas:** Fácil de crear, versionar y compartir entre usuarios y notebooks.
- **Gestión:** Unity Catalog permite controlar permisos y auditoría sobre estas funciones.

### 2. Funciones en Python/Scala (Spark UDFs)

Se crean usando código en Python o Scala y se registran en Spark:

python
from pyspark.sql.functions import udf

def mi_funcion(x):
    return x * 2

spark.udf.register("mi_funcion", mi_funcion)


- **Uso:** Se pueden usar en consultas SQL o DataFrames.
- **Limitación:** No se gestionan directamente en Unity Catalog (a menos que se empaqueten como funciones permanentes).

---

**Resumen:**  
- **Unity Catalog:** Permite crear y gestionar funciones SQL permanentes y seguras.
- **Spark UDFs:** Permiten lógica compleja en Python/Scala, pero su gestión es local al notebook o cluster.